# Expérience complète : apprendre, valider et recharger la même règle

Retrouver la première classification, puis vérifier que son fichier reproduit les scores.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#boucle-entrainement-complete).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Les trois extraits ont appris à être classés, et nous avons essayé des observations réservées. Rassemblons cette expérience pour pouvoir la rouvrir demain. Nous gardons X brut, le même Classifieur à deux entrées et deux sorties, les poids initiaux imposés et les 201 pas de SGD à 0,1.

Aucun nuage de points ni nouveau descripteur n’est nécessaire. Nos trois lignes tiennent dans un lot : le programme reste sur ce lot complet pour retrouver exactement la première boucle. Le Dataset devient utile lorsqu’il faut parcourir davantage d’exemples.

Le nombre rangé dans historique est obtenu par item : il sert au suivi, sans conserver un graphe de calcul. La perte affichée dans la boucle est celle d’avant le pas ; pour connaître la décision finale, on recalcule les scores après la dernière mise à jour.

| Mesures choisies : RMS/crête | Cible |
| --- | --- |
| [0,12 ; 1,8] | 0 : voix |
| [0,18 ; 4,0] | 1 : percussion |
| [0,16 ; 2,2] | 0 : voix |
| [0,22 ; 3,8] | 1 : percussion |

Ces quatre lignes choisies n’ont pas participé aux 201 pas. Elles ne constituent pas un test statistique sur des enregistrements réels.

La validation ne contient ni backward ni step. eval règle le comportement du modèle ; no_grad évite de construire un graphe pour cette mesure. Dans cette miniature, les trois exemples appris et les quatre réservés sont correctement classés. La perte de réserve reste positive : une classe correcte ne signifie pas une certitude parfaite.

Conservons maintenant la règle apprise. Le programme écrit classifieur-extraits.pt dans son dossier courant. state_dict contient ici linear.weight et linear.bias ; d’autres architectures peuvent aussi enregistrer des buffers persistants. Il ne contient pas la définition de Classifieur.

La copie commence avec d’autres valeurs, puis load_state_dict les remplace. map_location="cpu" choisit le placement au chargement ; weights_only=True convient à ce fichier d’état de tenseurs. On appelle encore eval : charger un état ne choisit pas le mode d’inférence.

Les scores rechargés sont identiques, pas seulement leurs classes. Nous avons conservé une règle de prédiction. Pour reprendre Adam avec ses moments et son compteur, il faudrait aussi son optimizer.state_dict() : cette autre tâche reste dans la référence sur les états.

## Démonstration 1 · La même classification, du premier pas au fichier

Retrouver les objets et exécuter les fragments du chapitre dans leur contexte complet.

### Préparer les bibliothèques

Classifieur organise la Linear déjà comprise. Les sept lignes sont choisies et la réserve reste séparée.

In [3]:
from pathlib import Path
import torch
from torch import nn


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [4]:
X = torch.tensor([[.1, 1.5], [.2, 4.5], [.15, 2.5]])
cibles = torch.tensor([0, 1, 0], dtype=torch.long)
X_reserve = torch.tensor([[.12, 1.8], [.18, 4.0], [.16, 2.2], [.22, 3.8]])
cibles_reserve = torch.tensor([0, 1, 0, 1], dtype=torch.long)



In [5]:
observer('X', X)
observer('cibles', cibles)
observer('X_reserve', X_reserve)
observer('cibles_reserve', cibles_reserve)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]
X_reserve forme (4, 2) type torch.float32 requires_grad False
  valeurs : [[0.11999999731779099, 1.7999999523162842], [0.18000000715255737, 4.0], [0.1599999964237213, 2.200000047683716], [0.2199999988079071, 3.799999952316284]]
cibles_reserve forme (4,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0, 1]


### Définir la règle avant de l’utiliser

Cette cellule définit un objet ou une fonction. Son corps ne sera utilisé que lors d’un appel. Nous gardons la définition entière pour ne pas casser la syntaxe.

In [6]:
class Classifieur(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 2)
    def forward(self, mesures):
        return self.linear(mesures)



### Calculer l’état suivant

Les poids et biais sont imposés, puis SGD et la perte sont préparés.

In [7]:
modele = Classifieur()


### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [8]:
with torch.no_grad():
    modele.linear.weight.copy_(torch.tensor([[0., 1.], [0., -1.]]))
    modele.linear.bias.zero_()


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [9]:
critere = nn.CrossEntropyLoss()
optimiseur = torch.optim.SGD(modele.parameters(), lr=.1)
historique = []


In [10]:
observer('historique', historique)

historique = []


### Calculer l’état suivant

Chaque coût est calculé avant sa mise à jour. item conserve seulement une valeur numérique.

In [11]:
modele.train()


Classifieur(
  (linear): Linear(in_features=2, out_features=2, bias=True)
)

### Répéter la même étape sur les données

Le corps indenté reste dans une seule cellule. Les opérations se répètent dans l’ordre écrit ; les sorties permettront de suivre leurs effets.

In [12]:
for pas in range(201):
    optimiseur.zero_grad()
    scores = modele(X)
    perte = critere(scores, cibles)
    perte.backward()
    optimiseur.step()
    historique.append(perte.item())



### Calculer l’état suivant

Les deux réglages de validation sont distincts et aucune mise à jour ne se trouve dans ce bloc.

In [13]:
modele.eval()


Classifieur(
  (linear): Linear(in_features=2, out_features=2, bias=True)
)

### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [14]:
with torch.no_grad():
    predictions_train = modele(X).argmax(dim=1)
    scores_reserve = modele(X_reserve)
    predictions_reserve = scores_reserve.argmax(dim=1)
    perte_reserve = critere(scores_reserve, cibles_reserve)


### Calculer l’état suivant

Le fichier conserve des tenseurs nommés ; la classe recrée leur architecture avant chargement.

In [15]:
fichier = Path("classifieur-extraits.pt")
torch.save(modele.state_dict(), fichier)
copie = Classifieur()
etat = torch.load(fichier, map_location="cpu", weights_only=True)


In [16]:
observer('fichier', fichier)
observer('copie', copie)
observer('etat', etat)

etat : clés ['linear.weight', 'linear.bias']


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [17]:
copie.load_state_dict(etat)


<All keys matched successfully>

### Calculer l’état suivant

L’état chargé n’active pas eval. On compare les scores sur les mêmes entrées réservées.

In [18]:
copie.eval()


Classifieur(
  (linear): Linear(in_features=2, out_features=2, bias=True)
)

### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [19]:
with torch.no_grad():
    scores_recharges = copie(X_reserve)


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [20]:
identiques = torch.equal(scores_reserve, scores_recharges)


In [21]:
observer('identiques', identiques)

identiques = True


### Lire les sorties

Les sorties décrivent cette expérience choisie, pas une mesure de généralisation audio.

In [22]:
print("pertes avant les pas 1 et 201 :", round(historique[0], 6), round(historique[-1], 6))
print("entrainement :", predictions_train.tolist())
print("reserve :", predictions_reserve.tolist(), "cibles", cibles_reserve.tolist())
print("perte reserve :", round(perte_reserve.item(), 6))


pertes avant les pas 1 et 201 : 3.018476 0.226087
entrainement : [0, 1, 0]
reserve : [0, 1, 0, 1] cibles [0, 1, 0, 1]
perte reserve : 0.309575


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [23]:
print("rechargement identique :", identiques)


rechargement identique : True


In [24]:
assert bool(len(historique) == 201 and historique[-1] < historique[0]), 'boucle-entrainement-complete/main: len(historique) == 201 and historique[-1] < historique[0]'
assert bool(predictions_train.tolist() == [0,1,0] and predictions_reserve.tolist() == [0,1,0,1]), 'boucle-entrainement-complete/main: predictions_train.tolist() == [0,1,0] and predictions_reserve.tolist() == [0,1,0,1]'
assert bool(not scores_reserve.requires_grad and not copie.training), 'boucle-entrainement-complete/main: not scores_reserve.requires_grad and not copie.training'
assert bool(identiques and torch.equal(scores_reserve, scores_recharges) and set(etat) == {"linear.weight","linear.bias"}), 'boucle-entrainement-complete/main: identiques and torch.equal(scores_reserve, scores_recharges) and set(etat) == {"linear.weight","linear.bias"}'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Les poids connus ont pour lignes [0,1] et [0,−1], avec biais nul. Prévoyez les trois décisions sur X, avant apprentissage.

In [25]:
exo_x=torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]])
exo_modele=torch.nn.Linear(2,2)
with torch.no_grad():
    exo_modele.weight.copy_(torch.tensor([[0.,1.],[0.,-1.]]))
    exo_modele.bias.zero_()
exo_decisions = ...  # Remplacer les points par votre réponse.
if exo_decisions is Ellipsis or exo_decisions is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.equal(exo_decisions,exo_modele(exo_x).argmax(dim=1))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Le premier score reste positif pour les trois lignes ; cela n’en fait pas les trois bonnes classes.

</details>

### Exercice 2

Rechargez l’état du classifieur sauvegardé, puis calculez ses scores sur ces deux descripteurs.

In [26]:
exo_modele=torch.nn.Linear(2,2)
with torch.no_grad():
    exo_modele.weight.copy_(torch.tensor([[0.,1.],[0.,-1.]]))
    exo_modele.bias.zero_()
exo_x=torch.tensor([[.2,4.5]])
torch.save(exo_modele.state_dict(),"exo-classifieur.pt")
exo_copie=torch.nn.Linear(2,2)
exo_etat=torch.load("exo-classifieur.pt",map_location="cpu",weights_only=True)
exo_copie.load_state_dict(exo_etat)
exo_copie.eval()
exo_scores = ...  # Remplacer les points par votre réponse.
if exo_scores is Ellipsis or exo_scores is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    torch.testing.assert_close(exo_scores,torch.tensor([[4.5,-4.5]]))
    assert torch.equal(exo_scores,exo_modele(exo_x))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Même architecture, état chargé, puis nouvelle passe. Charger des poids ne lance ni entraînement ni prédiction.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x=torch.tensor([[.1,1.5],[.2,4.5],[.15,2.5]])
exo_modele=torch.nn.Linear(2,2)
with torch.no_grad():
    exo_modele.weight.copy_(torch.tensor([[0.,1.],[0.,-1.]]))
    exo_modele.bias.zero_()
exo_decisions = torch.tensor([0,0,0])
if exo_decisions is Ellipsis or exo_decisions is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.equal(exo_decisions,exo_modele(exo_x).argmax(dim=1))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_modele=torch.nn.Linear(2,2)
with torch.no_grad():
    exo_modele.weight.copy_(torch.tensor([[0.,1.],[0.,-1.]]))
    exo_modele.bias.zero_()
exo_x=torch.tensor([[.2,4.5]])
torch.save(exo_modele.state_dict(),"exo-classifieur.pt")
exo_copie=torch.nn.Linear(2,2)
exo_etat=torch.load("exo-classifieur.pt",map_location="cpu",weights_only=True)
exo_copie.load_state_dict(exo_etat)
exo_copie.eval()
exo_scores = exo_copie(exo_x)
if exo_scores is Ellipsis or exo_scores is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    torch.testing.assert_close(exo_scores,torch.tensor([[4.5,-4.5]]))
    assert torch.equal(exo_scores,exo_modele(exo_x))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [29]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()